# TFM V2: frozen codebook token-map classifier

This is the first of three bounded follow-up versions after the failed V1 histogram probe. It reuses the existing subject/sentence token caches, preserves channel and temporal structure, freezes the official TFM tokenizer/codebook, and trains only a small temporal-convolution plus channel-attention classifier.

The independent evaluation unit remains the sentence. Reader recordings are separate training examples, each sentence has equal total loss weight, and reader probabilities are averaged for the held-out sentence score. The same known readers occur across folds on different sentences, so this tests unseen-sentence rather than unseen-subject generalization. Use a **GPU** Colab runtime.


## What changes from V1

| Component | V1 | V2 |
| --- | --- | --- |
| TFM | frozen tokenizer | frozen tokenizer and codebook |
| Feature | 8,192-bin histogram | full `channel x time` token map |
| Reader handling | feature-average before training | separate records; probability-average at test |
| Classifier | logistic regression | small temporal CNN + channel attention |
| Leakage control | sentence-level folds | sentence-level folds |

The run is resumable after every setup/fold. Do not change the architecture, seeds, filters, or thresholds after seeing results.


In [ ]:
# 1) Fetch this small codebase. Colab already supplies all V2 dependencies.
from pathlib import Path
import os, subprocess

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
os.chdir(PROJECT_ROOT / "tfm")
print("Working directory:", Path.cwd())

In [ ]:
# 2) Mount Drive. V2 reuses V1 tokens and writes to its own result folder.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/tfm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/tfm"
TOKEN_CACHE = CACHE_ROOT / "tokens_v1"
CHECKPOINT_CACHE = CACHE_ROOT / "upstream_checkpoints"
RESULTS_DIR = RESULTS_ROOT / "token_map_v2"

if not TOKEN_CACHE.exists():
    raise FileNotFoundError(f"V1 token cache not found: {TOKEN_CACHE}")
CHECKPOINT_CACHE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Token cache:", TOKEN_CACHE)
print("Checkpoint cache:", CHECKPOINT_CACHE)
print("V2 results:", RESULTS_DIR)

In [ ]:
# 3) Fetch official model source and persist only the selected tokenizer checkpoint.
import re, shutil, urllib.request

UPSTREAM_URL = "https://github.com/Jathurshan0330/TFM-Tokenizer.git"
UPSTREAM_ROOT = Path("/content/TFM-Tokenizer")
if not UPSTREAM_ROOT.exists():
    environment = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(UPSTREAM_ROOT)], env=environment)

def lfs_pointer(path):
    try:
        text = path.read_text()
    except (UnicodeDecodeError, OSError):
        return None
    match = re.search(r"^size (\d+)$", text, flags=re.MULTILINE)
    return int(match.group(1)) if text.startswith("version https://git-lfs") and match else None

weight_root = UPSTREAM_ROOT / "pretrained_weigths"  # upstream spelling
candidates = []
for path in weight_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
        name = str(path.relative_to(UPSTREAM_ROOT)).lower()
        if any(key in name for key in ("vq", "tokenizer", "tfm_token")) and not any(key in name for key in ("encoder", "classifier", "finetun")):
            candidates.append(path)
if not candidates:
    raise FileNotFoundError("No tokenizer checkpoint candidate found upstream")
candidates.sort(key=lambda path: ("multiple_dataset" not in str(path).lower(), "pretrain" not in path.name.lower(), len(str(path))))
source_checkpoint = candidates[0]
relative = source_checkpoint.relative_to(UPSTREAM_ROOT)
TOKENIZER_CHECKPOINT = CHECKPOINT_CACHE / relative
TOKENIZER_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
pointer_size = lfs_pointer(source_checkpoint)
if pointer_size is not None:
    if not TOKENIZER_CHECKPOINT.exists() or TOKENIZER_CHECKPOINT.stat().st_size != pointer_size:
        print(f"Caching checkpoint in Drive ({pointer_size / 2**20:.1f} MiB)")
        media_url = f"https://media.githubusercontent.com/media/Jathurshan0330/TFM-Tokenizer/master/{relative.as_posix()}"
        temporary = TOKENIZER_CHECKPOINT.with_suffix(TOKENIZER_CHECKPOINT.suffix + ".download")
        urllib.request.urlretrieve(media_url, temporary)
        if temporary.stat().st_size != pointer_size:
            raise IOError(f"Checkpoint size mismatch: expected {pointer_size}, got {temporary.stat().st_size}")
        temporary.replace(TOKENIZER_CHECKPOINT)
    else:
        print("Reusing Drive-cached checkpoint")
elif not TOKENIZER_CHECKPOINT.exists():
    shutil.copy2(source_checkpoint, TOKENIZER_CHECKPOINT)
upstream_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
print("Checkpoint:", TOKENIZER_CHECKPOINT)
print("Upstream revision:", upstream_revision)

In [ ]:
# 4) Load the frozen codebook and cached token maps, then run/resume V2.
import torch
from src.token_map import (
    TokenMapConfig,
    build_token_map_model,
    evaluate_token_map,
    extract_frozen_codebook_from_checkpoint,
    load_token_records,
)

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
config = TokenMapConfig()
codebook, codebook_report = extract_frozen_codebook_from_checkpoint(TOKENIZER_CHECKPOINT, config)
records, record_metadata, cache_report = load_token_records(TOKEN_CACHE, config)
probe = build_token_map_model(codebook, config)
trainable_parameters = sum(parameter.numel() for parameter in probe.parameters() if parameter.requires_grad)
print("Cache report:", cache_report)
print("Codebook report:", codebook_report)
print("Trainable classifier parameters:", trainable_parameters)
del probe

metrics, predictions, history, summary, delta, gate = evaluate_token_map(
    records=records,
    codebook=codebook,
    output_dir=RESULTS_DIR,
    cache_report=cache_report,
    codebook_report=codebook_report,
    config=config,
    device="cuda",
    resume=True,
)
display(summary)
print("Corrected paired bootstrap:", delta)
print("Decision:", gate["decision"])
print("Saved results:", RESULTS_DIR)

In [ ]:
# 5) Read the saved result record and create a reproducible per-seed figure.
import json
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
summary = pd.read_csv(RESULTS_DIR / "summary.csv", header=[0, 1])
gate = json.loads((RESULTS_DIR / "viability_gate.json").read_text())
display(summary)
seed_scores = metrics.groupby(["seed", "setup"])["macro_f1"].mean().unstack("setup")
display(seed_scores)
axes = seed_scores[["token_map", "token_map_shuffled", "majority"]].plot.bar(figsize=(8, 4))
axes.axhline(1 / 3, color="black", linestyle="--", linewidth=1, label="1/3 reference")
axes.set(title="TFM V2 macro-F1 by split seed", ylabel="macro-F1", xlabel="seed")
axes.legend(loc="best")
plt.tight_layout()
figure_path = RESULTS_DIR / "macro_f1_by_seed.png"
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(json.dumps(gate, indent=2))
print("Saved figure:", figure_path)

## Runtime and stopping rule

The first cache load can take several minutes because it reads about 4,500 compressed files from Drive. The 30 neural fits (aligned and shuffled, five folds, three seeds) can take roughly 1–3 hours depending on the Colab GPU. Every completed setup/fold is saved before the next starts; after a disconnect, rerun Cells 1–4 to resume, then Cell 5 when evaluation is complete.

V2 passes only if balanced accuracy is above one-third, aligned macro-F1 exceeds shuffled by at least 0.015, at least two seeds have positive deltas, the three-version-corrected 98.33% paired bootstrap lower bound is positive, and aligned macro-F1 beats majority. A failure proceeds only to the already planned V3; it does not authorize tuning V2.
